# Close out the experiments

Six remaining items. Four are cheap and exist because of what the previous
run found; two cost real time.

| Task | Why it is here | ~T4 time |
|---|---|---|
| 1. Latency | The last run failed with `No module named 'onnxruntime'`. Nothing was wrong with the exporter | 15 min |
| 2. Overlap, exact matching | The old radius of 5 matched 641 of 800 images, including 302 TB scans paired with `Normal-*` files. It was measuring anatomy, not identity | 5 min |
| 3. External under `simple` | Labels came back correct, so preprocessing is the open suspect for the sub-chance AUC | 5 min |
| 4. Probability dumps | The strongest result in the study, iterative pruning rescuing the 75% cliff, has no significance test because those checkpoints predate prob dumping | 3 min |
| 5. FLOPs | Structured pruning currently has a sparsity axis and no compute axis | 2 min |
| 6. `compact` at 30 epochs | `full` got 30 epochs and `compact` got 10. No capacity claim survives an unequal budget | 40 min |

**Attach:**

- `tawsifurrahman/tuberculosis-tb-chest-xray-dataset`
- `kmader/pulmonary-chest-xray-abnormalities`
- `wenhaolu49/notebook8807a976de` (the previous run: checkpoints and cache)

**GPU T4 x1**, **Internet on**. Tasks 1 to 5 total about 30 minutes; task 6 is
the only expensive one and has its own flag.

In [ ]:
import glob, json, os, shutil, subprocess, sys, time

REPO_URL = "https://github.com/AIscend-Research/lightweight-tb-net"
REPO_DIR = "/kaggle/working/lightweight-tb-net"

T1_LATENCY   = True
T2_OVERLAP   = True
T3_EXTERNAL  = True
T4_PROBS     = True
T5_FLOPS     = True
T6_COMPACT30 = True      # the expensive one
SEEDS        = [0, 1, 2, 3, 4]

ON_KAGGLE = os.path.exists("/kaggle/input")
if ON_KAGGLE and not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
elif not ON_KAGGLE:
    REPO_DIR = os.getcwd()
os.chdir(REPO_DIR)
SRC = os.path.join(REPO_DIR, "src")
COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True,
                        text=True).stdout.strip()
print("commit:", COMMIT)

# Task 1 exists entirely because these were missing last time.
# onnxruntime was absent in finish_run.ipynb; onnxscript is then the
# next blocker, because torch 2.6+ routes torch.onnx.export through
# the dynamo exporter by default and that path needs it.
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "onnx", "onnxruntime", "onnxscript"], check=True)
import onnxruntime
print("onnxruntime", onnxruntime.__version__)


def run(*cmd, check=True):
    cmd = [sys.executable] + list(cmd)
    print("$", " ".join(str(c) for c in cmd), flush=True)
    t0 = time.time()
    p = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in p.stdout:
        tail.append(line)
        print(line, end="")
    p.wait()
    print(f"[{(time.time() - t0) / 60:.1f} min, exit {p.returncode}]")
    if check and p.returncode != 0:
        raise RuntimeError("".join(tail[-40:]))
    return p.returncode


def reset(*names):
    """Clear a stage CSV before a forced re-run.

    Only safe when the command that follows regenerates every condition the
    file held. Last time this silently dropped the class-weighted arm from
    baseline.csv, so nothing here resets baseline.csv."""
    for n in names:
        f = os.path.join(REPO_DIR, "results", n)
        if os.path.exists(f):
            os.remove(f)
            print("cleared results/" + n)


def nomask(paths):
    return [p for p in paths if "mask" not in p.lower()]

In [ ]:
TB_DATASET    = "/kaggle/input/datasets/tawsifurrahman/tuberculosis-tb-chest-xray-dataset"
MC_SZ_DATASET = "/kaggle/input/datasets/kmader/pulmonary-chest-xray-abnormalities"
# Either previous run works: both carry the checkpoints and the image cache.
# Whichever is attached is used, most recent first.
PRIOR_RUNS = ["/kaggle/input/notebooks/wenhaolu49/notebook8807a976de",
              "/kaggle/input/notebooks/wenhaolu49/notebook7981e10859"]


def find_dir(root, must_contain):
    if not os.path.isdir(root):
        return None
    for dirpath, dirnames, _ in os.walk(root):
        if all(m in dirnames for m in must_contain):
            return dirpath
    return None


DATA_PATH = EXTERNAL_PATH = None
if ON_KAGGLE:
    DATA_PATH = (find_dir(TB_DATASET, ["Normal", "Tuberculosis"])
                 or find_dir("/kaggle/input", ["Normal", "Tuberculosis"]))
    hits = nomask(glob.glob(f"{MC_SZ_DATASET}/**/MCUCXR_*.png", recursive=True)
                  + glob.glob(f"{MC_SZ_DATASET}/**/CHNCXR_*.png", recursive=True))
    if hits:
        EXTERNAL_PATH = os.path.commonpath([os.path.dirname(h) for h in hits])
print("DATA_PATH     =", DATA_PATH)
print("EXTERNAL_PATH =", EXTERNAL_PATH)


def restore(src, dst):
    if src and os.path.isdir(src):
        shutil.copytree(src, os.path.join(REPO_DIR, dst), dirs_exist_ok=True)
        print(f"restored {dst}/ from {src}")
        return True
    return False


# Checkpoints and cache from the previous run. results/ comes from the repo,
# which already holds the merged and repaired copy.
candidates = [f"{r}/{sub}" for r in PRIOR_RUNS
              for sub in ("artifacts/checkpoints",
                          "lightweight-tb-net/checkpoints")]
candidates += sorted(glob.glob("/kaggle/input/**/checkpoints", recursive=True))
restore(next((d for d in candidates if os.path.isdir(d)), None), "checkpoints")

cache_candidates = [f"{r}/lightweight-tb-net/cache" for r in PRIOR_RUNS]
cache_candidates += sorted(glob.glob("/kaggle/input/**/cache", recursive=True))
restore(next((d for d in cache_candidates if os.path.isdir(d)), None), "cache")

n_ckpt = len(glob.glob(os.path.join(REPO_DIR, "checkpoints", "*.pth")))
have_cache = os.path.exists(f"{REPO_DIR}/cache/faithful.npy")
print(f"checkpoints: {n_ckpt} | cache: {have_cache}")

if DATA_PATH:
    run(f"{SRC}/make_splits.py", "--data-path", DATA_PATH,
        "--out", f"{REPO_DIR}/data_splits", "--seed", "42")
if not have_cache:
    assert DATA_PATH, "no cache and no dataset"
    run(f"{SRC}/build_cache.py", "--data-path", DATA_PATH, "--variant", "both")

## Task 1 - Latency

The previous failure was a missing package, not an exporter incompatibility.
With `onnxruntime` installed this should produce `results/latency.csv`.

Latency is a median of 100 CPU runs with its interquartile range. On a shared
cloud vCPU that is indicative, not device-representative, and the paper should
say so.

In [ ]:
# Tasks 1, 3, 4 and 5 all load trained models. If no previous run was
# attached, train the baselines they need first. A notebook cannot take its
# own output as an input, so this path is the normal one when you are editing
# the same notebook that produced the checkpoints.
if n_ckpt == 0:
    print("no checkpoints restored: training the baselines the later tasks "
          "need (~20 min)")
    run(f"{SRC}/experiments.py", "--stage", "baseline", "--force",
        "--arch", "compact", "full", "--caches", "faithful",
        "--seeds", *map(str, SEEDS), "--epochs", "10")
    n_ckpt = len(glob.glob(os.path.join(REPO_DIR, "checkpoints", "*.pth")))
    print(f"checkpoints now: {n_ckpt}")

if T1_LATENCY:
    reset("quantize.csv", "latency.csv", "latency_failures.csv")
    run(f"{SRC}/experiments.py", "--stage", "quantize", "--force",
        "--arch", "compact", "full", "--caches", "faithful",
        "--seeds", *map(str, SEEDS))

    import pandas as pd
    for f in ("latency.csv", "latency_failures.csv"):
        p = f"{REPO_DIR}/results/{f}"
        print(f"\n--- {f} ---")
        display(pd.read_csv(p)) if os.path.exists(p) else print("absent")

## Task 2 - Overlap, with a defensible threshold

The previous run reported 641 of 800 external images matching a training
image at Hamming distance 5 or less, and 302 of those paired a TB-positive
scan with a file named `Normal-*`. A 64-bit difference hash of a radiograph
downsampled to 9x8 encodes gross thoracic structure that every chest X-ray
shares, so that radius matches nearly anything.

This run takes the nearest neighbour rather than the first hit, defaults to
exact matching, and writes the full distance histogram to
`results/overlap_distances.csv`. Read the histogram: genuine duplicates show
up as a spike at or near zero separated by a gap. A smooth rise from small
distances means the hash cannot answer this question and no threshold on it
is trustworthy.

In [ ]:
if T2_OVERLAP and EXTERNAL_PATH:
    reset("overlap.csv", "overlap_distances.csv")
    run(f"{SRC}/experiments.py", "--stage", "overlap", "--force",
        "--external-path", EXTERNAL_PATH, "--caches", "faithful",
        "--hash-radius", "0")

    import matplotlib.pyplot as plt
    import pandas as pd
    p = f"{REPO_DIR}/results/overlap_distances.csv"
    if os.path.exists(p):
        d = pd.read_csv(p).sort_values("hamming")
        display(d)
        fig, ax = plt.subplots(figsize=(7, 3.5))
        ax.bar(d["hamming"], d["count"], color="#1f77b4")
        ax.set_xlabel("nearest-neighbour dHash Hamming distance")
        ax.set_ylabel("external images")
        ax.set_title("A spike near zero with a gap means real duplicates;\\n"
                     "a smooth rise means the hash is measuring anatomy")
        fig.savefig(f"{REPO_DIR}/figures/overlap_distance_hist.png", dpi=150,
                    bbox_inches="tight")
        plt.show()

## Task 3 - External validation under the `simple` pipeline

The label check came back clean (Montgomery 80/58, Shenzhen 326/336), so an
inverted convention does not explain the sub-chance AUC. Preprocessing is the
remaining suspect, and there is independent reason to look there: on the
internal test split `full` scores 98.29 sensitivity under `simple` and 88.29
under `faithful`. The paper's own pipeline appears to be hurting.

This evaluates the `simple`-trained models on external images processed the
same way. Results are appended with a `preproc` column so both pipelines sit
in one file and can be compared directly.

In [ ]:
if T3_EXTERNAL and EXTERNAL_PATH:
    reset("external.csv", "external_labels.csv")
    for preproc in ("faithful", "simple"):
        run(f"{SRC}/experiments.py", "--stage", "external", "--force",
            "--external-path", EXTERNAL_PATH, "--arch", "compact", "full",
            "--caches", preproc, "--seeds", *map(str, SEEDS))

    import pandas as pd
    e = pd.read_csv(f"{REPO_DIR}/results/external.csv")
    piv = e.groupby(["preproc", "arch", "cohort"])["auc"].agg(["mean", "std"])
    display(piv.round(4))
    print("\nAn AUC that crosses 0.5 when only the pipeline changes points at")
    print("preprocessing as the cause rather than domain shift.")

## Tasks 4 and 5 - Probability dumps and FLOPs

Both are inference-only and cost almost nothing.

The dumps give every existing checkpoint a `results/probs/` file, which is
what the iterative-versus-one-shot pruning comparison needs to get a
confidence interval.

FLOPs are reported dense and effective. Dense counts will be identical across
pruning conditions, because a zeroed weight is still multiplied; the effective
count charges only for output channels that are not entirely zero, which is
where structured pruning shows a real saving and unstructured pruning shows
none. Input-channel propagation is not modelled, so effective savings are
conservative.

In [ ]:
if T4_PROBS:
    run(f"{SRC}/experiments.py", "--stage", "dumpprobs", "--force",
        "--caches", "faithful")
    print("prob files:", len(glob.glob(f"{REPO_DIR}/results/probs/*.csv")))

if T5_FLOPS:
    reset("flops.csv")
    run(f"{SRC}/experiments.py", "--stage", "flops", "--force",
        "--arch", "compact", "full")

    import pandas as pd
    f = pd.read_csv(f"{REPO_DIR}/results/flops.csv")
    display(f.sort_values(["arch", "checkpoint"]).head(30))

## Task 6 - `compact` at 30 epochs

`full` was given 30 epochs and improved from 88.29 to 93.71 sensitivity, so it
was undertrained at 10. But `compact` has only ever been trained for 10, and a
capacity comparison across unequal budgets is not a comparison. This closes
that gap.

Note the append: `baseline.csv` is **not** reset here. It already holds the
class-weighted arm and the 30-epoch `full` arm, and a reset would delete both.

In [ ]:
# The guard above may already have written 10-epoch rows; the 30-epoch
# arm now saves under its own checkpoint name (_e30) so nothing is
# overwritten, and baseline.csv keeps both.
if T6_COMPACT30:
    run(f"{SRC}/experiments.py", "--stage", "baseline", "--force",
        "--arch", "compact", "--caches", "faithful",
        "--seeds", *map(str, SEEDS), "--epochs", "30")

    import pandas as pd
    b = pd.read_csv(f"{REPO_DIR}/results/baseline.csv")
    display(b.groupby(["arch", "cache", "weighted", "epochs"])[["sens", "acc"]]
            .agg(["mean", "std", "count"]).round(2))

## Regenerate everything and package

In [ ]:
json.dump({"commit": COMMIT, "repo": REPO_URL, "seeds": SEEDS,
           "archs": ["compact", "full"], "caches": ["faithful", "simple"],
           "data_path": DATA_PATH, "external_path": EXTERNAL_PATH,
           "tasks": {"latency": T1_LATENCY, "overlap": T2_OVERLAP,
                     "external_simple": T3_EXTERNAL, "dumpprobs": T4_PROBS,
                     "flops": T5_FLOPS, "compact30": T6_COMPACT30}},
          open(f"{REPO_DIR}/results/run_manifest.json", "w"), indent=2)

for f in glob.glob(f"{REPO_DIR}/results/summary_*.csv"):
    os.remove(f)
run(f"{SRC}/experiments.py", "--stage", "summary", check=False)
run(f"{SRC}/analyze.py")
run(f"{SRC}/make_figures.py", check=False)

from IPython.display import Markdown, display
display(Markdown(open(f"{REPO_DIR}/results/ANALYSIS.md").read()))

In [ ]:
# Small bundle only: results and figures. Checkpoints stay behind and belong
# on Zenodo.
OUT = "/kaggle/working/for_repo"
shutil.rmtree(OUT, ignore_errors=True)
for sub in ("results", "figures"):
    src = os.path.join(REPO_DIR, sub)
    if os.path.isdir(src):
        shutil.copytree(src, os.path.join(OUT, sub))
shutil.make_archive(OUT, "zip", OUT)
print(f"-> for_repo.zip ({os.path.getsize(OUT + '.zip') / 1024 ** 2:.1f} MB)")

for f in ("latency.csv", "overlap_distances.csv", "external.csv",
          "flops.csv", "baseline.csv", "history.csv"):
    p = os.path.join(OUT, "results", f)
    print(("  OK      " if os.path.exists(p) else "  absent  ") + f)

## Reading the output

1. `results/latency.csv` - if it exists, the deployment claim finally has
   measurement behind it. If `latency_failures.csv` appears instead, read the
   error rather than theorising about it.
2. `figures/overlap_distance_hist.png` - a spike at zero separated by a gap
   means genuine duplicates and the external evaluation is compromised. A
   smooth rise means the hash cannot answer the question and the honest write
   up is that the overlap remains untested.
3. `results/external.csv` grouped by `preproc` - if AUC crosses 0.5 when only
   the pipeline changes, the sub-chance result was a preprocessing artefact.
4. `results/flops.csv` - expect identical dense counts everywhere and a real
   effective saving only for the structured rows.
5. `baseline.csv` at 30 epochs for both architectures - the first honest
   capacity comparison in the study.